# Evaluation visualization

Read completed evaluation outputs and save five figures and readable tables. No model weights, contextual extraction or metric scoring are needed. Run the cells in order in a fresh kernel. Figures are saved as PNG at 300 dpi and EPS; they are not displayed inline.


## 1. Parameters

Choose the input outputs directory and a new presentation ID. Shared fonts, dimensions and figure names follow the embedded project protocol.


In [ ]:
from pathlib import Path


def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src" / "phonetics").is_dir():
            return candidate
        child = candidate / "project"
        if (child / "src" / "phonetics").is_dir():
            return child
    raise FileNotFoundError("Could not find project root")

PROJECT_ROOT = find_project_root()

# The directory containing contextual-anchor-q/, contextual-dimsim/ and contextual-random-control/.
EVALUATION_OUTPUT_ROOT = PROJECT_ROOT / "outputs"

# None selects the only completed run per analysis. Multiple completed runs require explicit IDs.
# EVALUATION_RUN_IDS = {
#     "contextual-anchor-q": "evaluation-01",
#     "contextual-dimsim": "evaluation-01",
#     "contextual-random-control": "evaluation-01",
# }
EVALUATION_RUN_IDS = None

# Results are written to OUTPUT_PROJECT_ROOT/outputs/contextual-evaluation-report/PRESENTATION_ID/.
OUTPUT_PROJECT_ROOT = PROJECT_ROOT / "outputs"
PRESENTATION_ID = "duyu-chinese-lessons-zh-cn-v1"

# Leave empty when Arial is installed. Otherwise provide existing Arial regular and bold font files.
FONT_FILES = []
# FONT_FILES = [Path("/kaggle/input/your-fonts/arial.ttf"), Path("/kaggle/input/your-fonts/arialbd.ttf")]

INSTALL_DEPENDENCIES = False
DEPENDENCIES = ["numpy", "matplotlib"]


## 2. Dependencies

Only NumPy and Matplotlib are required beyond the Python standard library. Package installation is optional.


In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import tempfile
from contextlib import contextmanager
from dataclasses import dataclass
from hashlib import sha256
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Any

if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11 or newer is required")
import tomllib

if "EDIT_ME" in str(EVALUATION_OUTPUT_ROOT):
    raise ValueError("Set EVALUATION_OUTPUT_ROOT in the parameter cell")
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *DEPENDENCIES])

_plot_cache = tempfile.TemporaryDirectory(prefix="evaluation-matplotlib-")
os.environ["MPLCONFIGDIR"] = _plot_cache.name
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager

print("numpy", np.__version__)
print("matplotlib", matplotlib.__version__)


In [ ]:
BUILD_METADATA = {
    "schema_version": 1,
    "scope": "evaluation_visualization",
    "source_sha256": {
        "research/artifacts.py": "663ec77c515bcbe1f350aa12b2d4fea80491ef10512069be387845f8c522c239",
        "evaluation/evaluation_outputs.py": "520962f59222ef4927251e6e848522d6769427a924139189982ece217d068355",
        "evaluation/evaluation_tables.py": "5e1c5c3ff81e15aff0d958b6f2f58afe2f1c711c090837935ce9cece0c83b638",
        "visualization/core.py": "181904bd8f9f4436ee8df96b3f594e2f6926f89d37a64a9c4c45eb4e698b84df",
        "visualization/evaluation.py": "5da2ea572631e918778eb5255bba492afa0805e742c4ba4b45c56fc17704d8ed",
        "visualization/evaluation_report.py": "ffdcebeba90bb67781769930f09ed4d1eb5bad70fd91ffd0439075c7a80b2391"
    },
    "resource_sha256": {
        "visualization/style.toml": "5dcaf9f879cbe9629cc2670f7d6a5f1ab80f3271a032624274fa05210043f677",
        "visualization/registry.toml": "c558bdf83abef0e83a615a11252a8dd89d75dbe555927b9507d836e4d1425c51"
    },
    "definition_sha256": {
        "research/artifacts.py": "4d17b9f045852b756ea23a2312aa2de35cf712d004151819fe852710e3538f86",
        "evaluation/evaluation_outputs.py": "9fe229d6e6426908499da4079f145f22bc0070f601b51a3474cbb7f92788d1de",
        "evaluation/evaluation_tables.py": "6d7a25ed47b4c64b239657a6b60e415cbf50f8f9445e9bfa47ca37770f8591dc",
        "visualization/core.py": "6dfe3290985b5c2cac199cc629b2c4755d570c4ff8b1779620c3138c8a3016ed",
        "visualization/evaluation.py": "4df4cec41155b4036e42756b3bb4f6296cf6a9c6aa32f956981e9eb6113d2901",
        "visualization/evaluation_report.py": "b4aa645da784caab3b409060a0462a65f66165a13ac5404e7596f30a23e5ac25"
    },
    "generator_sha256": "266fcbfc77aeaab71f49d7418bc5b740dd65756f2ebf9d935b4806296efcb419"
}


## 3. Arial and figure settings

Use installed Arial or provide font files in the parameter cell. Font substitution and automatic font downloads are disabled.


In [ ]:
for filename in FONT_FILES:
    filename = Path(filename).resolve(strict=True)
    font_manager.fontManager.addfont(str(filename))
if not any(font.name == "Arial" for font in font_manager.fontManager.ttflist):
    raise RuntimeError("Arial is unavailable. Set FONT_FILES to existing Arial regular and bold files, or run in an environment with Arial installed.")

_style_resources = tempfile.TemporaryDirectory(prefix="evaluation-style-")
STYLE_RESOURCE_ROOT = Path(_style_resources.name)


In [ ]:
STYLE_TOML = r'''[schema]
version = 1

[font]
family = "Arial"
axis_label_pt = 9.0
tick_pt = 8.0
legend_pt = 8.0
panel_heading_pt = 9.0
panel_letter_pt = 10.0
annotation_pt = 8.0
normal_weight = "normal"
panel_letter_weight = "bold"

[color]
background = "#FFFFFF"
foreground = "#171717"
secondary = "#666666"
tertiary = "#888888"
quaternary = "#999999"

[line]
data_pt = 1.2
axes_pt = 0.7
reference_pt = 0.7
interval_pt = 0.9
caps_pt = 3.0

[marker]
size_pt = 4.0
edge_pt = 0.8
face = "white"
markevery_target = 10

[layout]
single_width_mm = 90.0
multi_width_mm = 180.0
panel_aspect = 1.3333333333333333
left_mm = 14.0
right_mm = 5.0
bottom_mm = 12.0
top_mm = 8.0
hgap_mm = 14.0
vgap_mm = 16.0
legend_row_mm = 8.0
legend_gap_mm = 3.0
default_columns = [1, 2, 2, 2, 3, 3, 3]

[axes]
grid = false
minor_ticks = false
tick_direction = "in"
major_length_pt = 3.0
major_width_pt = 0.7
major_pad_pt = 3.0
major_target = 5
panel_label_x_mm = -3.0
panel_label_y_mm = 2.0

[legend]
max_per_row = 4
handle_length_em = 2.4
column_spacing_em = 1.2
location = "upper center"

[histogram]
density = true
fill = false
bootstrap_bins = 40

[export]
formats = ["png", "eps"]
dpi = 300
transparent = false
overwrite = false

[backend.rcparams]
"pdf.fonttype" = 42
"ps.fonttype" = 42
"svg.fonttype" = "none"
'''
_ = (STYLE_RESOURCE_ROOT / 'style.toml').write_text(STYLE_TOML, encoding='utf-8')

REGISTRY_TOML = r'''[schema]
version = 1

[figures.pilot_ecdf]
status = "approved"
kind = "ecdf"
name = "Cosine distributions"
xlabel = "Cosine similarity"
ylabel = "ECDF"
series = ["base", "zh_cn"]
order = ["base", "zh_cn"]
scale = "linear"
xlim = [-1.0, 1.0]
ylim = [0.0, 1.0]

[figures.pilot_histogram]
status = "approved"
kind = "histogram"
name = "Cosine density"
xlabel = "Cosine similarity"
ylabel = "Density"
series = ["base", "zh_cn"]
order = ["base", "zh_cn"]
scale = "linear"
density = true
xlim = [-1.0, 1.0]
ylim = [0.0]
edges_start = -1.0
edges_stop = 1.0
edges_count = 61

[figures.pilot_bootstrap]
status = "approved"
kind = "bootstrap_histogram"
name = "Bootstrap differences"
xlabel = "Zh-cn \u2212 Base"
ylabel = "Bootstrap repeats"
series = ["delta"]
order = ["delta"]
scale = "linear"
bins = 40
density = false
ylim = [0.0]

[semantics.base]
label = "Base"
color = "#171717"
linestyle = "solid"
marker = "o"

[semantics.zh_cn]
label = "Zh-cn"
color = "#666666"
linestyle = "dashed"
marker = "s"

[semantics.delta]
label = "Zh-cn \u2212 Base"
color = "#171717"
linestyle = "solid"
marker = "o"

[figures.dimsim_distributions]
status = "approved"
kind = "distribution_triptych"
name = "DIMSIM distributions"
xlabel = "DIMSIM distance"
ylabel = "Pair count / ECDF"
series = ["population"]
order = ["population"]
scale = "linear"

[figures.dimsim_overlap_distributions]
status = "approved"
kind = "ecdf"
name = "Component overlap distributions"
xlabel = "DIMSIM distance"
ylabel = "ECDF"
series = ["000", "001", "010", "011", "100", "101", "110", "111"]
order = ["000", "001", "010", "011", "100", "101", "110", "111"]
scale = "linear"
ylim = [0.0, 1.0]

[figures.dimsim_component_contributions]
status = "approved"
kind = "stacked_bar"
name = "DIMSIM component contributions"
xlabel = "DIMSIM interval"
ylabel = "Share of mean DIMSIM distance"
series = ["initial", "final", "tone"]
order = ["initial", "final", "tone"]
scale = "linear"

[semantics.population]
label = "All pairs"
color = "#171717"
linestyle = "solid"
marker = "o"

[semantics.000]
label = "000"
color = "#171717"
linestyle = "solid"
marker = "o"
[semantics.001]
label = "001"
color = "#444444"
linestyle = "dashed"
marker = "s"
[semantics.010]
label = "010"
color = "#666666"
linestyle = "dotted"
marker = "^"
[semantics.011]
label = "011"
color = "#888888"
linestyle = "dashdot"
marker = "v"
[semantics.100]
label = "100"
color = "#303030"
linestyle = "solid"
marker = "D"
[semantics.101]
label = "101"
color = "#606060"
linestyle = "dashed"
marker = "P"
[semantics.110]
label = "110"
color = "#909090"
linestyle = "dotted"
marker = "X"
[semantics.111]
label = "111"
color = "#c0c0c0"
linestyle = "dashdot"
marker = "*"
[semantics.initial]
label = "Initial"
color = "#303030"
linestyle = "solid"
marker = "o"
[semantics.final]
label = "Final"
color = "#888888"
linestyle = "solid"
marker = "s"
[semantics.tone]
label = "Tone"
color = "#d0d0d0"
linestyle = "solid"
marker = "^"

[figures.anchor_q_transitions]
status = "approved"
kind = "dotplot"
name = "Anchor-Q transitions"
xlabel = "Transition"
ylabel = "Anchor-Q"
series = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
order = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
scale = "linear"

[figures.transition_shares]
status = "approved"
kind = "stacked_bar"
name = "Transition shares"
xlabel = "Transition"
ylabel = "Q share"
series = ["initial", "final", "tone"]
order = ["initial", "final", "tone"]
scale = "linear"

[figures.component_shares]
status = "approved"
kind = "stacked_bar"
name = "Component shares"
xlabel = "Model"
ylabel = "Q share"
series = ["initial", "final", "tone"]
order = ["initial", "final", "tone"]
scale = "linear"

[figures.dimsim_geometry]
status = "approved"
kind = "line_histogram"
name = "DIMSIM geometry"
xlabel = "DIMSIM distance"
ylabel = "Cosine similarity"
series = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
order = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
scale = "linear"

[figures.dimsim_correlation]
status = "approved"
kind = "dotplot"
name = "DIMSIM correlation"
xlabel = "Model"
ylabel = "Spearman rho"
series = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
order = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]
scale = "linear"

[semantics.syllable]
label = "Syllable"
color = "#171717"
linestyle = "solid"
marker = "o"
[semantics.components]
label = "Components"
color = "#444444"
linestyle = "dashed"
marker = "s"
[semantics.acoustic]
label = "Acoustic"
color = "#707070"
linestyle = "dotted"
marker = "^"
[semantics."components-acoustic-lambda-1"]
label = "Combined \u03bb1"
color = "#999999"
linestyle = "dashdot"
marker = "D"
[semantics."components-acoustic-lambda-10"]
label = "Combined \u03bb10"
color = "#c5c5c5"
linestyle = "solid"
marker = "v"
'''
_ = (STYLE_RESOURCE_ROOT / 'registry.toml').write_text(REGISTRY_TOML, encoding='utf-8')



### Input and output paths


In [ ]:
def file_sha256(path, chunk=1024 * 1024):
    h = sha256()
    with Path(path).open('rb') as f:
        while (data := f.read(chunk)):
            h.update(data)
    return h.hexdigest()

def safe_run_id(value):
    if not isinstance(value, str) or not re.fullmatch('[A-Za-z0-9][A-Za-z0-9._-]{0,127}', value) or value in {'.', '..'}:
        raise ValueError('run_id must be a non-empty safe filename component')
    return value

def safe_analysis(value):
    if not isinstance(value, str) or not re.fullmatch('[a-z0-9][a-z0-9-]{0,63}', value):
        raise ValueError('analysis must be a safe lowercase name')
    return value

def ensure_output_outside_inputs(output, input_roots):
    output = Path(output).resolve()
    for root in map(lambda p: Path(p).resolve(), input_roots):
        if output == root or root in output.parents:
            raise ValueError(f'Output must not be inside input: {root}')


### Verified artifact loading and output manifests


In [ ]:
def data_json_bytes(value: Any) -> bytes:
    return (json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_data_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('xb') as stream:
        stream.write(data_json_bytes(value))

def artifact_files(root: Path) -> set[str]:
    return {p.relative_to(root).as_posix() for p in root.rglob('*') if p.is_file()}

def read_finite_json(path: Path):

    def bad(token):
        raise ValueError(f'non-finite JSON constant: {token}')
    value = json.loads(path.read_text(encoding='utf-8'), parse_constant=bad)
    data_json_bytes(value)
    return value

def check_input_pins(inputs, scientific_pins):
    bundle_pin = scientific_pins.get('bundle_manifest_sha256')
    if bundle_pin and bundle_pin not in {x['sha256'] for x in inputs}:
        raise ValueError('scientific bundle manifest pin does not match an input identity')
    exp_pin = scientific_pins.get('experiment_manifest_sha256')
    if exp_pin and exp_pin not in {x['sha256'] for x in inputs}:
        raise ValueError('scientific experiment manifest pin does not match an input identity')

def verify_artifact_inventory(root: Path, analysis: str, run_id: str, expected: set[str], *, profile: str='numerical_outputs', figure_exports: str='pending'):
    safe_analysis(analysis)
    safe_run_id(run_id)
    manifest = read_finite_json(root / 'manifest.json')
    if manifest.get('schema_version') != 1 or manifest.get('status') != 'complete' or manifest.get('analysis') != analysis or (manifest.get('run_id') != run_id) or (manifest.get('scope') != profile) or (manifest.get('figure_exports') != figure_exports) or (set(manifest.get('required_artifacts', [])) != expected):
        raise ValueError('output manifest identity/profile mismatch')
    if not isinstance(manifest.get('python'), str) or not manifest['python'] or (not isinstance(manifest.get('dependencies'), dict)) or ('numpy' not in manifest['dependencies']) or (not isinstance(manifest.get('inputs'), list)) or (not isinstance(manifest.get('source_hashes'), dict)) or (not manifest['source_hashes']):
        raise ValueError('output provenance metadata is incomplete')
    for row in manifest['inputs']:
        if not isinstance(row, dict) or not isinstance(row.get('path'), str) or (not isinstance(row.get('identity_path'), str)) or (not is_sha256(row.get('sha256'))):
            raise ValueError('output input identity metadata is invalid')
    if any((not is_sha256(x) for x in manifest['source_hashes'].values())):
        raise ValueError('output source hash metadata is invalid')
    actual, inventory = (artifact_files(root) - {'manifest.json'}, manifest.get('artifacts'))
    declared = {x.get('path') for x in inventory if isinstance(x, dict)} if isinstance(inventory, list) else set()
    if actual != expected or declared != expected or len(inventory or []) != len(expected):
        raise ValueError('output artifact inventory mismatch')
    for row in inventory:
        if not isinstance(row, dict) or set(row) != {'path', 'sha256', 'bytes'}:
            raise ValueError('malformed artifact inventory row')
        path = root / row['path']
        if not path.is_file() or path.stat().st_size <= 0 or path.stat().st_size != row['bytes'] or (file_sha256(path) != row['sha256']):
            raise ValueError(f"artifact hash or size mismatch: {row.get('path')}")
    return manifest

def is_sha256(value):
    return isinstance(value, str) and len(value) == 64 and all((c in '0123456789abcdef' for c in value))

@contextmanager
def presentation_run(project_root: str | Path, analysis: str, run_id: str, *, input_roots: list[str | Path], config: dict, source_hashes: dict, expected_files: set[str], scientific_pins: dict, create_data: bool=True, output_path: str | Path | None=None, profile: str='numerical_outputs', figure_exports_on_complete: str='pending'):
    """Write one run, gating every file except its top-level manifest."""
    analysis, run_id = (safe_analysis(analysis), safe_run_id(run_id))
    if figure_exports_on_complete not in {'pending', 'complete'}:
        raise ValueError('unsupported figure export completion status')
    root = Path(output_path).resolve() if output_path is not None else Path(project_root).resolve() / 'outputs' / analysis / run_id
    ensure_output_outside_inputs(root, input_roots)
    if root.exists():
        raise FileExistsError(f'refusing to overwrite existing output directory: {root}')
    if not isinstance(config, dict) or not isinstance(source_hashes, dict) or (not source_hashes):
        raise ValueError('explicit config and source hash map are required')
    if any((not isinstance(v, str) or len(v) != 64 or any((c not in '0123456789abcdef' for c in v)) for v in source_hashes.values())):
        raise ValueError('source hashes must be lowercase SHA-256 digests')
    data_json_bytes({'config': config, 'source_hashes': source_hashes, 'scientific_pins': scientific_pins})
    inputs = []
    for raw in input_roots:
        p = Path(raw).resolve(strict=True)
        identity = p if p.is_file() else p / 'manifest.json'
        if not identity.is_file():
            raise FileNotFoundError(f'input manifest identity missing: {identity}')
        inputs.append({'path': str(p), 'identity_path': str(identity), 'sha256': file_sha256(identity)})
    check_input_pins(inputs, scientific_pins)
    dependencies = {}
    for package in ('numpy', 'scipy', 'pandas', 'matplotlib'):
        try:
            dependencies[package] = version(package)
        except PackageNotFoundError:
            dependencies[package] = None
    root.mkdir(parents=True, exist_ok=False)
    manifest = {'schema_version': 1, 'analysis': analysis, 'run_id': run_id, 'status': 'running', 'scope': profile, 'figure_exports': 'pending', 'required_artifacts': sorted(expected_files), 'artifacts': [], 'scientific_input_pins': scientific_pins, 'source_hashes': source_hashes, 'inputs': inputs, 'python': platform.python_version(), 'dependencies': dependencies}
    write_data_json(root / 'manifest.json', manifest)
    try:
        names = (('data',) if create_data else ()) + ('tables', 'figures', 'diagnostics')
        for name in names:
            (root / name).mkdir()
        write_data_json(root / 'config.json', config)
        yield root
        actual = artifact_files(root) - {'manifest.json'}
        if actual != expected_files:
            raise RuntimeError(f'artifact inventory mismatch; missing={sorted(expected_files - actual)}, unexpected={sorted(actual - expected_files)}')
        artifacts = []
        for relative in sorted(actual):
            path = root / relative
            if path.stat().st_size <= 0:
                raise RuntimeError(f'required artifact is empty: {relative}')
            artifacts.append({'path': relative, 'sha256': file_sha256(path), 'bytes': path.stat().st_size})
        manifest.update(status='complete', artifacts=artifacts, figure_exports=figure_exports_on_complete)
    except BaseException as exc:
        manifest.update(status='failed', error={'type': type(exc).__name__, 'message': str(exc)})
        raise
    finally:
        fd, name = tempfile.mkstemp(prefix='manifest-', suffix='.tmp', dir=root)
        try:
            with os.fdopen(fd, 'wb') as stream:
                stream.write(data_json_bytes(manifest))
            os.replace(name, root / 'manifest.json')
        finally:
            if os.path.exists(name):
                os.unlink(name)

def load_evaluation_outputs(root: str | Path) -> dict:
    """Load a complete numerical run after checking exact inventory and hashes."""
    root = Path(root).resolve(strict=True)
    manifest_path = root / 'manifest.json'
    analysis = root.parent.name
    if analysis not in {'contextual-anchor-q', 'contextual-dimsim'}:
        raise ValueError('unknown evaluation output analysis')
    q = analysis.endswith('anchor-q')
    bn, rn = ('anchor_q_batch.json', 'anchor_q_report.json') if q else ('dimsim_batch.json', 'dimsim_report.json')
    expected = {'config.json', f'data/{bn}', f'data/{rn}', 'tables/tables.json', 'figures/plot_data.json', 'diagnostics/summary.json'}
    manifest = verify_artifact_inventory(root, analysis, root.name, expected)
    check_input_pins(manifest.get('inputs', []), manifest.get('scientific_input_pins', {}))
    if set(manifest['scientific_input_pins']) != {'bundle_manifest_sha256', 'model_ids', 'seeds', 'model_seed_provenance', 'analysis', 'run_id'}:
        raise ValueError('numerical output pins are malformed')
    batch, report = (read_finite_json(root / f'data/{bn}'), read_finite_json(root / f'data/{rn}'))
    tables, plot = (read_finite_json(root / 'tables/tables.json'), read_finite_json(root / 'figures/plot_data.json'))
    config = read_finite_json(root / 'config.json')
    read_finite_json(root / 'diagnostics/summary.json')
    if tables != report.get('tables') or plot != report.get('plot_data'):
        raise ValueError('stored tables or plot data differ from stored report')
    if batch.get('bundle_manifest_sha256') != manifest['scientific_input_pins']['bundle_manifest_sha256']:
        raise ValueError('stored batch bundle pin mismatch')
    results = batch.get('results')
    pins = manifest['scientific_input_pins']
    if not isinstance(results, list) or sorted({r.get('model_id') for r in results if isinstance(r, dict)}) != pins['model_ids']:
        raise ValueError('stored batch model inventory differs from manifest pins')
    if any((not isinstance(r, dict) or type(r.get('seed')) is not int or r.get('seed') not in pins['seeds'] for r in results)):
        raise ValueError('stored batch seed inventory is invalid')
    identity_pins = {(p.get('model_id'), p.get('seed')): p for p in pins['model_seed_provenance'] if isinstance(p, dict)}
    expected_pairs = {(model, seed) for model in pins['model_ids'] for seed in pins['seeds']}
    if len(identity_pins) != len(pins['model_seed_provenance']) or set(identity_pins) != expected_pairs:
        raise ValueError('manifest model/seed provenance inventory is invalid')
    for row in results:
        pin = identity_pins[row['model_id'], row['seed']]
        if any((row.get(k) != pin.get(k) for k in ('run_id', 'run_manifest_sha256', 'checkpoint_sha256'))):
            raise ValueError('stored batch provenance differs from manifest pins')
        if row.get('experiment_hashes') is not None and row['experiment_hashes'] != pin.get('experiment_hashes'):
            raise ValueError('stored batch experiment hashes differ from manifest pins')
    if config.get('reference_model') is not None and report.get('reference_model') is not None and (report.get('reference_model') != config['reference_model']):
        raise ValueError('stored report reference metadata differs from config')
    return {'root': root, 'manifest': manifest, 'config': config, 'batch': batch, 'report': report, 'tables': tables, 'plot_data': plot}

def load_random_control_summary(root: str | Path) -> dict:
    """Read hash-verified random-control summaries without loading vector matrices."""
    root = Path(root).resolve(strict=True)
    expected = {'config.json', 'data/occurrence_ids.json', 'data/control_report.json', 'tables/tables.json', 'figures/plot_data.json', 'diagnostics/summary.json', *(f'data/random_seed_{seed}.npy' for seed in range(5))}
    manifest = verify_artifact_inventory(root, 'contextual-random-control', root.name, expected)
    pins = manifest.get('scientific_input_pins', {})
    check_input_pins(manifest.get('inputs', []), pins)
    if set(pins) != {'bundle_manifest_sha256', 'seed_kind', 'random_seeds', 'dimension'} or pins['seed_kind'] != 'random' or pins['random_seeds'] != list(range(5)) or (type(pins['dimension']) is not int) or (pins['dimension'] <= 0):
        raise ValueError('random-control scientific pins malformed')
    report = read_finite_json(root / 'data/control_report.json')
    if report.get('schema_version') != 1 or report.get('model_id') != 'random-control' or report.get('seed_kind') != 'random' or (report.get('random_seeds') != list(range(5))) or (report.get('bundle_manifest_sha256') != pins['bundle_manifest_sha256']):
        raise ValueError('stored random-control identity mismatch')
    tables = read_finite_json(root / 'tables/tables.json')
    plot = read_finite_json(root / 'figures/plot_data.json')
    if tables != {'anchor_q': report.get('anchor_q'), 'dimsim': report.get('dimsim')} or plot != tables:
        raise ValueError('random-control table or plot payload mismatch')
    return {'root': root, 'manifest': manifest, 'config': read_finite_json(root / 'config.json'), 'report': {key: value for key, value in report.items() if key != 'per_seed'}, 'tables': tables, 'plot_data': plot}


### Readable table export


In [ ]:
def guard_table_output(target):
    path = Path(target).resolve()
    for ancestor in (path, *path.parents):
        manifest = ancestor / 'manifest.json'
        if not manifest.is_file():
            continue
        try:
            data = json.loads(manifest.read_text(encoding='utf-8'))
        except (OSError, ValueError):
            continue
        if data.get('scope') == 'numerical_outputs' and data.get('analysis') in {'contextual-anchor-q', 'contextual-dimsim'} and (data.get('status') in {'complete', 'failed'}):
            raise ValueError(f'Refusing to write inside immutable evaluation run: {ancestor}')

def format_table_number(v):
    return '—' if v is None else str(v) if isinstance(v, int) and (not isinstance(v, bool)) else f'{v:.3f}'

def markdown_table(title, headers, rows):
    out = [f'# {title}', '', '| ' + ' | '.join(headers) + ' |', '|' + '|'.join(('---' for _ in headers)) + '|']
    out += ['| ' + ' | '.join((str(x) for x in row)) + ' |' for row in rows]
    return '\n'.join(out) + '\n'

def render_evaluation_tables(anchor_tables: dict, dimsim_tables: dict, output_dir):
    """Write Markdown tables from report tables without recalculating summaries."""
    out = Path(output_dir)
    guard_table_output(out)
    if out.exists() and any(out.iterdir()):
        raise FileExistsError('Table output directory must be empty')
    out.mkdir(parents=True, exist_ok=True)
    artifacts = {}

    def write(name, body):
        if Path(name).name != name or not name.endswith('.md'):
            raise ValueError('Unsafe table filename')
        (out / name).write_text(body, encoding='utf-8')
        artifacts[name] = body
    models = sorted({r['model_id'] for r in anchor_tables['model_seed_edges']})
    for model in models:
        for seed in range(5):
            rows = [r for r in anchor_tables['model_seed_edges'] if r['model_id'] == model and r['seed'] == seed]
            fields = ('edge_id', 'q', 'mc_se', 'draw_count', 'unique_triplets', 'unique_anchor_occurrences', 'unique_anchor_syllables', 'unique_occurrences')
            write(f'anchor_q_edges_{model}_seed_{seed}.md', markdown_table(f'Anchor-Q edges: {model}, seed {seed}', list(fields), [[r['edge_id'], format_table_number(r['q']), format_table_number(r['mc_se']), r['draw_count'], *[(r.get('coverage') or {}).get(k, '—') for k in fields[4:]]] for r in rows]))
        metric_rows = [r for r in anchor_tables['model_seed_metrics'] if r['model_id'] == model]
        summary = anchor_tables.get('model_metric_seed_summary', [])
        fields = ('metric', 'seed', 'q', 'mc_se')
        write(f'anchor_q_metrics_{model}.md', markdown_table(f'Anchor-Q metrics: {model}', list(fields), [[r['metric'], r['seed'], format_table_number(r['q']), format_table_number(r['mc_se'])] for r in metric_rows]))
        sm = [r for r in summary if r['model_id'] == model]
        write(f'anchor_q_metric_summary_{model}.md', markdown_table(f'Anchor-Q mean and sample SD: {model}', ['Metric', 'Mean', 'Sample SD', 'MC SE'], [[r['metric'], format_table_number(r['mean']), format_table_number(r['sample_sd']), format_table_number(r['mc_se'])] for r in sm]))
        erows = [r for r in anchor_tables.get('model_edge_seed_summary', []) if r['model_id'] == model]
        write(f'anchor_q_edge_summary_{model}.md', markdown_table(f'Anchor-Q edge summary: {model}', ['Edge', 'Mean', 'Sample SD', 'MC SE'], [[r['edge_id'], format_table_number(r['mean']), format_table_number(r['sample_sd']), format_table_number(r['mc_se'])] for r in erows]))
        metric_names = ('initial', 'final', 'tone', 'overall')
        by_seed = {seed: {r['metric']: format_table_number(r['q']) for r in anchor_tables['model_seed_metrics'] if r['model_id'] == model and r['seed'] == seed} for seed in range(5)}
        smap = {r['metric']: r for r in anchor_tables.get('model_metric_seed_summary', []) if r['model_id'] == model}
        rows = [[f'Seed {seed}'] + [by_seed[seed].get(metric, '—') for metric in metric_names] for seed in range(5)]
        rows.append(['Mean ± sample SD'] + [f"{format_table_number(smap[m]['mean'])} ± {format_table_number(smap[m]['sample_sd'])}" for m in metric_names if m in smap])
        write(f'anchor_q_metrics_wide_{model}.md', markdown_table(f'Anchor-Q metrics by seed: {model}', ['Seed', *metric_names], rows))
    for key, title, fields in (('model_seed_edges', 'Anchor-Q edge values', ('model_id', 'seed', 'edge_id', 'q', 'mc_se', 'draw_count')), ('model_seed_metrics', 'Anchor-Q metric values', ('model_id', 'seed', 'metric', 'q', 'mc_se')), ('global_metric_summary', 'Anchor-Q global summaries', ('model_id', 'metric', 'mean', 'sample_sd', 'mc_se')), ('paired_deltas', 'Paired Anchor-Q differences', ('model_id', 'reference_model', 'metric', 'mean_delta_q', 'sample_sd_seed_delta', 'mc_se')), ('transition_shares', 'Anchor-Q transition shares', ('model_id', 'edge_id', 'share', 'status')), ('component_shares', 'Anchor-Q component shares', ('model_id', 'component', 'share', 'status'))):
        rows = [[r.get(f) if isinstance(r.get(f), str) else format_table_number(r.get(f)) for f in fields] for r in anchor_tables[key]]
        write(f'{key}.md', markdown_table(title, list(fields), rows))
    paired = []
    for r in anchor_tables['paired_deltas']:
        paired.append([r['model_id'], r['metric'], f"{format_table_number(r['mean_delta_q'])} ({format_table_number(r['sample_sd_seed_delta'])}; {format_table_number(r['mc_se'])})"])
    write('anchor_q_paired_delta_cells.md', markdown_table('Paired Anchor-Q differences', ['Model', 'Metric', 'Mean ΔQ (sample SD; MC SE)'], paired))
    metric_names = ('initial', 'final', 'tone', 'overall')
    paired_map = {(r['model_id'], r['metric']): r for r in anchor_tables['paired_deltas']}
    paired_models = sorted({r['model_id'] for r in anchor_tables['paired_deltas']})
    paired_wide = [[model] + [f"{format_table_number(paired_map[model, m]['mean_delta_q'])} ({format_table_number(paired_map[model, m]['sample_sd_seed_delta'])}; {format_table_number(paired_map[model, m]['mc_se'])})" for m in metric_names] for model in paired_models]
    write('anchor_q_paired_deltas_wide.md', markdown_table('Paired Anchor-Q differences', ['Model', *metric_names], paired_wide))
    global_map = {(r['model_id'], r['metric']): r for r in anchor_tables['global_metric_summary']}
    global_models = sorted({r['model_id'] for r in anchor_tables['global_metric_summary']})
    global_wide = [[model] + [f"{format_table_number(global_map[model, m]['mean'])} ± {format_table_number(global_map[model, m]['sample_sd'])}" for m in metric_names] + [format_table_number(global_map[model, m]['mc_se']) for m in metric_names] for model in global_models]
    write('anchor_q_global_summary_wide.md', markdown_table('Anchor-Q global summary', ['Model', *[f'{m} mean ± SD' for m in metric_names], *[f'{m} MC SE' for m in metric_names]], global_wide))
    for key, title, fields in (('seed_rho', 'DIMSIM full-range rho by seed', ('model_id', 'seed', 'mode', 'rho', 'valid_draw_count', 'unique_pair_count', 'status')), ('model_rho', 'DIMSIM full-range rho summaries', ('model_id', 'mode', 'mean', 'sample_sd', 'status')), ('paired_rho_deltas', 'Paired DIMSIM rho differences', ('model_id', 'reference_model', 'mode', 'mean', 'sample_sd', 'status')), ('seed_boundary', 'Exploratory boundary thresholds', ('model_id', 'seed', 'mode', 'threshold', 'status')), ('bins', 'DIMSIM curve bins and support', ('model_id', 'mode', 'bin', 'mean_cosine', 'supported'))):
        rows = []
        for r in dimsim_tables[key]:
            row = []
            for f in fields:
                v = r.get(f)
                if f == 'bin' and isinstance(r.get('seed_points'), list):
                    pts = r['seed_points']
                    v = f"{v}; draws={pts[0].get('draw_count')}; unique={pts[0].get('unique_pair_count')}" if pts else v
                row.append(v if isinstance(v, (str, bool)) else format_table_number(v))
            rows.append(row)
        write(f'{key}.md', markdown_table(title, list(fields), rows))
    if 'model_boundary' in dimsim_tables:
        rows = [[r.get('model_id'), r.get('mode'), format_table_number(r.get('mean')), format_table_number(r.get('sample_sd')), format_table_number(r.get('median')), r.get('status')] for r in dimsim_tables['model_boundary']]
        write('dimsim_boundary_summary.md', markdown_table('Exploratory DIMSIM boundary summary', ['Model', 'Mode', 'Mean', 'Sample SD', 'Median', 'Status'], rows))
    if 'model_rho' in dimsim_tables:
        modes = sorted({r['mode'] for r in dimsim_tables['model_rho']})
        rho_map = {(r['model_id'], r['mode']): r for r in dimsim_tables['model_rho']}
        dmodels = sorted({r['model_id'] for r in dimsim_tables['model_rho']})
        wide = [[model] + [f"{format_table_number(rho_map[model, mode]['mean'])} ± {format_table_number(rho_map[model, mode]['sample_sd'])}" for mode in modes] for model in dmodels]
        write('dimsim_model_rho_wide.md', markdown_table('DIMSIM full-range rho by model and mode', ['Model', *modes], wide))
    return sorted(out.glob('*.md'))


### Shared figure protocol


In [ ]:
ROOT = STYLE_RESOURCE_ROOT
STYLE_PATH = ROOT / 'style.toml'
REGISTRY_PATH = ROOT / 'registry.toml'
MM_PER_INCH = 25.4

class VisualizationError(ValueError):
    """An actionable protocol violation."""
STYLE_KEYS = {'schema': {'version'}, 'font': {'family', 'axis_label_pt', 'tick_pt', 'legend_pt', 'panel_heading_pt', 'panel_letter_pt', 'annotation_pt', 'normal_weight', 'panel_letter_weight'}, 'color': {'background', 'foreground', 'secondary', 'tertiary', 'quaternary'}, 'line': {'data_pt', 'axes_pt', 'reference_pt', 'interval_pt', 'caps_pt'}, 'marker': {'size_pt', 'edge_pt', 'face', 'markevery_target'}, 'layout': {'single_width_mm', 'multi_width_mm', 'panel_aspect', 'left_mm', 'right_mm', 'bottom_mm', 'top_mm', 'hgap_mm', 'vgap_mm', 'legend_row_mm', 'legend_gap_mm', 'default_columns'}, 'axes': {'grid', 'minor_ticks', 'tick_direction', 'major_length_pt', 'major_width_pt', 'major_pad_pt', 'major_target', 'panel_label_x_mm', 'panel_label_y_mm'}, 'legend': {'max_per_row', 'handle_length_em', 'column_spacing_em', 'location'}, 'histogram': {'density', 'fill', 'bootstrap_bins'}, 'export': {'formats', 'dpi', 'transparent', 'overwrite'}, 'backend': {'rcparams'}}
FIGURE_KEYS = {'status', 'kind', 'name', 'xlabel', 'ylabel', 'series', 'order', 'scale', 'bin_rule', 'bins', 'density', 'columns', 'xlim', 'ylim', 'edges_start', 'edges_stop', 'edges_count'}
SEMANTIC_KEYS = {'label', 'color', 'linestyle', 'marker'}
ALLOWED_RCPARAMS = {'pdf.fonttype', 'ps.fonttype', 'svg.fonttype'}

def read_style_toml(p):
    raw = Path(p).read_bytes()
    return (tomllib.loads(raw.decode()), raw)

def reject_unknown_style_keys(d, a, w):
    e = set(d) - a
    if e:
        raise VisualizationError(f"Unknown key(s) in {w}: {', '.join(sorted(e))}")

def require_positive_style_value(v, n):
    if isinstance(v, bool) or not isinstance(v, (int, float)) or (not math.isfinite(v)) or (v <= 0):
        raise VisualizationError(f'{n} must be finite and positive')

@dataclass(frozen=True)
class Protocol:
    style: dict[str, Any]
    registry: dict[str, Any]
    style_hash: str
    registry_hash: str
    style_path: Path
    registry_path: Path

    def figure(self, fid):
        try:
            return self.registry['figures'][fid]
        except KeyError as e:
            raise VisualizationError(f'Unknown figure ID: {fid}') from e

def load_protocol(style_path=STYLE_PATH, registry_path=REGISTRY_PATH):
    sp = Path(style_path).resolve()
    rp = Path(registry_path).resolve()
    s, sr = read_style_toml(sp)
    r, rr = read_style_toml(rp)
    reject_unknown_style_keys(s, set(STYLE_KEYS), 'style root')
    for sec, keys in STYLE_KEYS.items():
        if sec not in s:
            raise VisualizationError(f'Missing style section: {sec}')
        reject_unknown_style_keys(s[sec], keys, f'style.{sec}')
        missing = keys - set(s[sec])
        if missing:
            raise VisualizationError(f"Missing key(s) in style.{sec}: {', '.join(sorted(missing))}")
    if s['schema']['version'] != 1:
        raise VisualizationError('Unsupported style schema version')
    if s['font']['family'] != 'Arial':
        raise VisualizationError('The protocol requires Arial')
    for k, v in s['font'].items():
        if k.endswith('_pt'):
            require_positive_style_value(v, f'font.{k}')
    if (s['font']['normal_weight'], s['font']['panel_letter_weight']) != ('normal', 'bold'):
        raise VisualizationError('Font weights must be normal and bold')
    for k, v in s['layout'].items():
        if k != 'default_columns':
            require_positive_style_value(v, f'layout.{k}')
    if s['layout']['default_columns'] != [1, 2, 2, 2, 3, 3, 3]:
        raise VisualizationError('Invalid default columns')
    a = s['axes']
    if a['grid'] is not False or a['minor_ticks'] is not False or a['tick_direction'] != 'in':
        raise VisualizationError('Axes require grid/minor false and inward ticks')
    for k in ('major_length_pt', 'major_width_pt', 'major_pad_pt', 'major_target'):
        require_positive_style_value(a[k], f'axes.{k}')
    if s['export']['formats'] != ['png', 'eps'] or s['export']['dpi'] != 300 or s['export']['transparent'] is not False:
        raise VisualizationError('Export requires opaque PNG300 then EPS')
    reject_unknown_style_keys(s['backend']['rcparams'], ALLOWED_RCPARAMS, 'style.backend.rcparams')
    reject_unknown_style_keys(r, {'schema', 'figures', 'semantics'}, 'registry root')
    if r.get('schema', {}).get('version') != 1 or not isinstance(r.get('figures'), dict) or (not isinstance(r.get('semantics'), dict)):
        raise VisualizationError('Invalid registry schema')
    for sid, sem in r['semantics'].items():
        reject_unknown_style_keys(sem, SEMANTIC_KEYS, f'semantics.{sid}')
        if set(sem) != SEMANTIC_KEYS or not all((isinstance(x, str) and x.strip() for x in sem.values())):
            raise VisualizationError(f'Invalid semantics.{sid}')
    for fid, f in r['figures'].items():
        reject_unknown_style_keys(f, FIGURE_KEYS, f'figures.{fid}')
        req = {'status', 'kind', 'name', 'xlabel', 'ylabel', 'series', 'order', 'scale'}
        if req - set(f):
            raise VisualizationError(f'Missing required registry keys for {fid}')
        if f['status'] not in {'draft', 'approved'} or f['kind'] not in {'ecdf', 'histogram', 'bootstrap_histogram', 'distribution_triptych', 'stacked_bar', 'dotplot', 'line_histogram'} or f['scale'] not in {'linear', 'log'}:
            raise VisualizationError(f'Invalid status/kind/scale for {fid}')
        if not all((isinstance(f[k], str) and f[k].strip() for k in ('name', 'xlabel', 'ylabel'))):
            raise VisualizationError(f'Invalid labels for {fid}')
        if f['series'] != f['order'] or not f['series'] or len(set(f['series'])) != len(f['series']) or set(f['series']) - set(r['semantics']):
            raise VisualizationError(f'Invalid series/order for {fid}')
        if 'bin_rule' in f and f['bin_rule'] != 'pooled_fd_sturges':
            raise VisualizationError(f'Invalid bin_rule for {fid}')
        if 'bins' in f and (isinstance(f['bins'], bool) or not isinstance(f['bins'], int) or f['bins'] < 1):
            raise VisualizationError(f'Invalid bins for {fid}')
        if 'xlim' in f and (not isinstance(f['xlim'], list) or len(f['xlim']) != 2 or (not all((isinstance(x, (int, float)) and (not isinstance(x, bool)) and math.isfinite(x) for x in f['xlim']))) or (f['xlim'][0] >= f['xlim'][1])):
            raise VisualizationError(f'Invalid xlim for {fid}')
        if f['scale'] == 'log' and 'xlim' in f and (f['xlim'][0] <= 0):
            raise VisualizationError(f'Log xlim must be positive for {fid}')
        if f['kind'] == 'ecdf':
            if 'ylim' not in f or not isinstance(f['ylim'], list) or len(f['ylim']) != 2 or (not all((isinstance(x, (int, float)) and (not isinstance(x, bool)) and math.isfinite(x) for x in f['ylim']))) or (f['ylim'][0] >= f['ylim'][1]):
                raise VisualizationError(f'ECDF ylim must be two ascending finite values for {fid}')
        elif f['kind'] in {'histogram', 'bootstrap_histogram'}:
            if not isinstance(f.get('density'), bool):
                raise VisualizationError(f'Histogram density must be boolean for {fid}')
            y = f.get('ylim')
            if not isinstance(y, list) or len(y) not in {1, 2} or (not all((isinstance(x, (int, float)) and (not isinstance(x, bool)) and math.isfinite(x) for x in y))) or (len(y) == 2 and y[0] >= y[1]):
                raise VisualizationError(f'Histogram ylim must be a finite bottom or ascending pair for {fid}')
            if f['kind'] == 'histogram' and (not ({'edges_start', 'edges_stop', 'edges_count'} <= set(f) or f.get('bin_rule') == 'pooled_fd_sturges')):
                raise VisualizationError(f'Histogram requires fixed edges or pooled bin rule for {fid}')
            if f['kind'] == 'bootstrap_histogram' and 'bins' not in f:
                raise VisualizationError(f'Bootstrap histogram requires bins for {fid}')
        edgekeys = {'edges_start', 'edges_stop', 'edges_count'} & set(f)
        if edgekeys and (edgekeys != {'edges_start', 'edges_stop', 'edges_count'} or not all((isinstance(f[k], (int, float)) and (not isinstance(f[k], bool)) and math.isfinite(f[k]) for k in ('edges_start', 'edges_stop'))) or f['edges_stop'] <= f['edges_start'] or isinstance(f['edges_count'], bool) or (not isinstance(f['edges_count'], int)) or (f['edges_count'] < 2)):
            raise VisualizationError(f'Invalid edge specification for {fid}')
    return Protocol(s, r, sha256(sr).hexdigest(), sha256(rr).hexdigest(), sp, rp)

def verify_style_integrity(p):
    if sha256(p.style_path.read_bytes()).hexdigest() != p.style_hash or sha256(p.registry_path.read_bytes()).hexdigest() != p.registry_hash:
        raise VisualizationError('Protocol files changed; reload')
    if tomllib.loads(p.style_path.read_text()) != p.style or tomllib.loads(p.registry_path.read_text()) != p.registry:
        raise VisualizationError('Protocol mappings were mutated; reload')

def require_arial():
    from matplotlib import font_manager
    if not any((f.name == 'Arial' for f in font_manager.fontManager.ttflist)):
        raise VisualizationError('Arial is not installed; font fallback is prohibited')

def apply_style(p):
    import matplotlib as mpl
    require_arial()
    s = p.style
    f = s['font']
    a = s['axes']
    c = s['color']
    rc = {'font.family': f['family'], 'font.weight': f['normal_weight'], 'font.size': f['tick_pt'], 'axes.labelsize': f['axis_label_pt'], 'axes.titlesize': f['panel_heading_pt'], 'axes.labelweight': f['normal_weight'], 'axes.titleweight': f['normal_weight'], 'xtick.labelsize': f['tick_pt'], 'ytick.labelsize': f['tick_pt'], 'legend.fontsize': f['legend_pt'], 'axes.linewidth': s['line']['axes_pt'], 'lines.linewidth': s['line']['data_pt'], 'lines.markersize': s['marker']['size_pt'], 'lines.markeredgewidth': s['marker']['edge_pt'], 'lines.markerfacecolor': s['marker']['face'], 'axes.edgecolor': c['foreground'], 'axes.labelcolor': c['foreground'], 'text.color': c['foreground'], 'xtick.color': c['foreground'], 'ytick.color': c['foreground'], 'figure.facecolor': c['background'], 'axes.facecolor': c['background'], 'axes.grid': a['grid'], 'xtick.direction': a['tick_direction'], 'ytick.direction': a['tick_direction'], 'xtick.top': False, 'ytick.right': False, 'xtick.minor.visible': False, 'ytick.minor.visible': False, 'xtick.major.size': a['major_length_pt'], 'ytick.major.size': a['major_length_pt'], 'xtick.major.width': a['major_width_pt'], 'ytick.major.width': a['major_width_pt'], 'xtick.major.pad': a['major_pad_pt'], 'ytick.major.pad': a['major_pad_pt'], 'axes.spines.left': True, 'axes.spines.right': True, 'axes.spines.top': True, 'axes.spines.bottom': True}
    rc.update(s['backend']['rcparams'])
    mpl.rcParams.update(rc)

def layout(n, p, *, columns=None, legend_rows=0, width_mm=None):
    import matplotlib.pyplot as plt
    from matplotlib.ticker import MaxNLocator
    if not isinstance(n, int) or not 1 <= n <= 26:
        raise VisualizationError('Panel count must be 1..26')
    l = p.style['layout']
    cols = columns if columns is not None else l['default_columns'][n - 1] if n <= 7 else 3
    if not isinstance(cols, int) or not 1 <= cols <= min(3, n):
        raise VisualizationError('columns must be 1..min(3,n)')
    if width_mm is not None:
        require_positive_style_value(width_mm, 'width_mm')
        width = float(width_mm)
    else:
        width = l['single_width_mm'] if cols == 1 else l['multi_width_mm']
    rows = (n + cols - 1) // cols
    pw = (width - l['left_mm'] - l['right_mm'] - (cols - 1) * l['hgap_mm']) / cols
    if pw <= 0:
        raise VisualizationError('width_mm is too small for physical margins')
    ph = pw / l['panel_aspect']
    leg = legend_rows * l['legend_row_mm'] + (l['legend_gap_mm'] if legend_rows else 0)
    height = l['bottom_mm'] + l['top_mm'] + leg + rows * ph + (rows - 1) * l['vgap_mm']
    fig = plt.figure(figsize=(width / MM_PER_INCH, height / MM_PER_INCH))
    axes = []
    a = p.style['axes']
    for i in range(n):
        row = i // cols
        nr = min(cols, n - row * cols)
        col = i % cols
        x = l['left_mm'] + (cols - nr) * (pw + l['hgap_mm']) / 2 + col * (pw + l['hgap_mm'])
        y = l['bottom_mm'] + (rows - 1 - row) * (ph + l['vgap_mm'])
        ax = fig.add_axes([x / width, y / height, pw / width, ph / height])
        axes.append(ax)
        ax.tick_params(top=False, right=False, bottom=True, left=True, direction='in', length=a['major_length_pt'], width=a['major_width_pt'], pad=a['major_pad_pt'])
        ax.minorticks_off()
        ax.grid(False)
        ax.xaxis.set_major_locator(MaxNLocator(a['major_target']))
        ax.yaxis.set_major_locator(MaxNLocator(a['major_target']))
        ax.text(a['panel_label_x_mm'] / pw, 1 + a['panel_label_y_mm'] / ph, chr(97 + i), transform=ax.transAxes, fontsize=p.style['font']['panel_letter_pt'], fontweight=p.style['font']['panel_letter_weight'], va='bottom')
    return (fig, axes)

def finite(values, *, policy='raise'):
    if policy not in {'raise', 'drop'}:
        raise VisualizationError('finite_policy must be raise or drop')
    a = np.asarray(values, float)
    m = np.isfinite(a)
    if not m.all() and policy == 'raise':
        raise VisualizationError('Non-finite values present; select drop explicitly')
    a = a[m]
    if not len(a):
        raise VisualizationError('No finite observations')
    return a

def ecdf(values, *, finite_policy='raise'):
    x = np.sort(finite(values, policy=finite_policy), kind='mergesort')
    return (x, np.arange(1, len(x) + 1) / len(x))

def pooled_edges(series, *, finite_policy='raise'):
    x = np.concatenate([finite(v, policy=finite_policy) for v in series])
    if x.min() == x.max():
        return np.array([x[0] - 0.5, x[0] + 0.5])
    q = np.percentile(x, [25, 75])
    iqr = q[1] - q[0]
    bins = int(np.ceil((x.max() - x.min()) / (2 * iqr / np.cbrt(len(x))))) if iqr > 0 else int(np.ceil(np.log2(len(x))) + 1)
    return np.linspace(x.min(), x.max(), max(1, bins) + 1)

def _record(fig, d):
    fig._phonetics_resolution = d

def plot_ecdf(ax, series, figure, p, *, finite_policy='raise'):
    counts = {}
    for sid in figure['order']:
        x, y = ecdf(series[sid], finite_policy=finite_policy)
        counts[sid] = len(x)
        sem = p.registry['semantics'][sid]
        ax.step(x, y, where='post', label=sem['label'], color=sem['color'], linestyle=sem['linestyle'], marker=None, linewidth=p.style['line']['data_pt'])
    ax.set(xlabel=figure['xlabel'], ylabel=figure['ylabel'], ylim=tuple(figure['ylim']), xscale=figure['scale'])
    ax.set_xlim(tuple(figure['xlim'])) if 'xlim' in figure else None
    _record(ax.figure, {'kind': 'ecdf', 'finite_policy': finite_policy, 'counts': counts, 'limits': {'x': list(ax.get_xlim()), 'y': list(ax.get_ylim())}})

def plot_histogram(ax, series, figure, p, *, edges=None, finite_policy='raise'):
    arr = [finite(series[s], policy=finite_policy) for s in figure['order']]
    edges = np.linspace(figure['edges_start'], figure['edges_stop'], figure['edges_count']) if edges is None and 'edges_count' in figure else edges
    edges = np.histogram_bin_edges(np.concatenate(arr), bins=figure['bins']) if edges is None and 'bins' in figure else pooled_edges(arr) if edges is None else np.asarray(edges, float)
    edges = np.asarray(edges, float)
    if len(edges) < 2 or not np.all(np.isfinite(edges)) or (not np.all(np.diff(edges) > 0)):
        raise VisualizationError('Histogram edges must be finite and strictly increasing')
    for sid, x in zip(figure['order'], arr):
        sem = p.registry['semantics'][sid]
        ax.hist(x, bins=edges, density=figure['density'], histtype='step', label=sem['label'], color=sem['color'], linestyle=sem['linestyle'], linewidth=p.style['line']['data_pt'])
    ax.set(xlabel=figure['xlabel'], ylabel=figure['ylabel'], xscale=figure['scale'])
    ax.set_ylim(tuple(figure['ylim'])) if len(figure['ylim']) == 2 else ax.set_ylim(bottom=figure['ylim'][0])
    ax.set_xlim(tuple(figure['xlim'])) if 'xlim' in figure else None
    _record(ax.figure, {'kind': figure['kind'], 'histogram_edges': edges.tolist(), 'finite_policy': finite_policy, 'counts': {s: len(x) for s, x in zip(figure['order'], arr)}, 'limits': {'x': list(ax.get_xlim()), 'y': list(ax.get_ylim())}})
    return edges

def shared_legend(fig, axes, p, *, rows=1):
    h, l = axes[0].get_legend_handles_labels()
    c = p.style['legend']
    return fig.legend(h, l, loc=c['location'], ncol=min(c['max_per_row'], max(1, int(np.ceil(len(l) / rows)))), frameon=False, handlelength=c['handle_length_em'], columnspacing=c['column_spacing_em'])

def export_figure(fig, output_dir, stem, figure_id, p, *, scientific_provenance, overwrite=None):
    verify_style_integrity(p)
    spec = p.figure(figure_id)
    if spec['status'] != 'approved':
        raise VisualizationError(f'Figure {figure_id!r} is draft; approval required')
    if not re.fullmatch('[A-Za-z0-9][A-Za-z0-9._-]*', stem) or stem in {'.', '..'}:
        raise VisualizationError('Unsafe export stem')
    prov = dict(scientific_provenance)
    if not re.fullmatch('[0-9a-fA-F]{64}', str(prov.get('source_digest', ''))):
        raise VisualizationError('source_digest must be SHA-256 hex')
    require_arial()
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    if not out.is_dir():
        raise VisualizationError('Export target is not a directory')
    paths = [out / f'{stem}.{x}' for x in p.style['export']['formats']]
    mp = out / f'{stem}.metadata.json'
    allow = p.style['export']['overwrite'] if overwrite is None else overwrite
    if not allow and any((x.exists() for x in paths + [mp])):
        raise FileExistsError('Refusing to overwrite export or metadata')
    from matplotlib.colors import to_rgba_array
    for a in fig.findobj():
        if hasattr(a, 'get_visible') and (not a.get_visible()):
            continue
        alpha = a.get_alpha() if hasattr(a, 'get_alpha') else None
        if alpha is not None and 0 < alpha < 1:
            raise VisualizationError('EPS forbids partial transparency')
        for getter in ('get_color', 'get_markerfacecolor', 'get_markeredgecolor', 'get_facecolor', 'get_edgecolor'):
            if not hasattr(a, getter):
                continue
            try:
                rgba = to_rgba_array(getattr(a, getter)())
            except (TypeError, ValueError):
                continue
            if rgba.size and np.any((rgba[:, 3] > 0) & (rgba[:, 3] < 1)):
                raise VisualizationError('EPS forbids RGBA partial transparency')
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    box = fig.get_window_extent(renderer)
    from matplotlib.legend import Legend
    from matplotlib.text import Text
    ticks = set()
    for ax in fig.axes:
        for axis in (ax.xaxis, ax.yaxis):
            lo, hi = sorted(axis.get_view_interval())
            tol = max(1, abs(hi - lo)) * 1e-12
            for tick in axis._update_ticks():
                loc = tick.get_loc()
                if math.isfinite(loc) and lo - tol <= loc <= hi + tol:
                    ticks.update((x for x in (tick.label1, tick.label2) if x.get_visible()))
    for a in fig.findobj(lambda x: isinstance(x, (Text, Legend))):
        if isinstance(a, Text) and a not in ticks and any((a is z for ax in fig.axes for z in ax.get_xticklabels() + ax.get_yticklabels())):
            continue
        if not a.get_visible():
            continue
        b = a.get_window_extent(renderer)
        if b.width and b.height and (b.x0 < box.x0 - 1 or b.y0 < box.y0 - 1 or b.x1 > box.x1 + 1 or (b.y1 > box.y1 + 1)):
            raise VisualizationError('Visible text or legend is clipped')
    meta = {'schema_version': 1, 'config_schema_version': p.style['schema']['version'], 'figure_id': figure_id, 'style_hash': p.style_hash, 'registry_hash': p.registry_hash, 'resolved_style': p.style, 'resolved_figure': spec, 'resolved_data': getattr(fig, '_phonetics_resolution', {}), 'scientific_provenance': prov}
    temps = []
    try:
        for dest in paths:
            t = dest.with_name(dest.stem + '.tmp' + dest.suffix)
            temps.append(t)
            import matplotlib as mpl
            with mpl.rc_context({'savefig.bbox': None, 'savefig.pad_inches': 0, 'savefig.dpi': p.style['export']['dpi'], 'savefig.facecolor': p.style['color']['background'], 'savefig.transparent': False}):
                fig.savefig(t, format=dest.suffix[1:], dpi=p.style['export']['dpi'], bbox_inches=None, pad_inches=0, transparent=False, facecolor=p.style['color']['background'], metadata={'Creator': 'phonetics.visualization'})
        mt = mp.with_name(mp.stem + '.tmp' + mp.suffix)
        temps.append(mt)
        mt.write_text(json.dumps(meta, indent=2, sort_keys=True))
        for t, d in zip(temps, paths + [mp]):
            os.replace(t, d)
    finally:
        for t in temps:
            if t.exists():
                t.unlink()
    return paths


### Five evaluation figures


In [ ]:
MODEL_LABELS = {'syllable': 'Syllable', 'components': 'Components', 'acoustic': 'Acoustic', 'components-acoustic-lambda-1': 'Combined λ1', 'components-acoustic-lambda-10': 'Combined λ10'}
FIGURE_STEMS = ('anchor-q-transitions', 'transition-shares', 'component-shares', 'dimsim-geometry', 'dimsim-correlation')

def model_semantics(p, model):
    return p.registry['semantics'][model]

def manifest_digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_plot_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def guard_report_output(target):
    path = Path(target).resolve()
    for ancestor in (path, *path.parents):
        manifest = ancestor / 'manifest.json'
        if not manifest.is_file():
            continue
        try:
            data = read_plot_json(manifest)
        except (OSError, ValueError):
            continue
        if str(data.get('analysis', '')).startswith('contextual-') and data.get('status') in {'complete', 'failed'}:
            raise ValueError(f'Refusing to write inside immutable evaluation run: {ancestor}')

def check_paired_outputs(aq, ds):
    am, dm = (aq['manifest'], ds['manifest'])
    if am.get('analysis') != 'contextual-anchor-q' or dm.get('analysis') != 'contextual-dimsim':
        raise ValueError('output roles must be anchor-Q and DIMSIM respectively')
    ap, dp = (am.get('scientific_input_pins', {}), dm.get('scientific_input_pins', {}))
    if ap.get('analysis') != 'contextual-anchor-q' or dp.get('analysis') != 'contextual-dimsim':
        raise ValueError('scientific output roles do not match manifests')
    for key in ('bundle_manifest_sha256', 'model_ids', 'seeds'):
        if ap.get(key) != dp.get(key):
            raise ValueError(f'paired outputs differ in {key}')

    def checkpoints(pins):
        rows = pins.get('model_seed_provenance')
        if not isinstance(rows, list):
            raise ValueError('model/seed checkpoint provenance is required')
        result = {(x.get('model_id'), x.get('seed')): x for x in rows if isinstance(x, dict)}
        if len(result) != len(rows) or not result or any((not re.fullmatch('[0-9a-f]{64}', str(x.get('checkpoint_sha256', ''))) for x in result.values())):
            raise ValueError('invalid model/seed checkpoint provenance')
        return result
    if checkpoints(ap) != checkpoints(dp):
        raise ValueError('paired outputs use different model/seed checkpoint provenance')

def build_figures(anchor_plot, dimsim_plot, protocol=None):
    """Build figures from the two persisted report plot_data objects."""
    p = protocol or load_protocol()
    figs = []
    models = list(MODEL_LABELS)
    try:
        fig, axs = layout(3, p, columns=3, legend_rows=2, width_mm=180)
        figs.append(fig)
        components = ('initial', 'final', 'tone')
        labels = None
        rows = anchor_plot['edges']
        for ci, (ax, component) in enumerate(zip(axs, components)):
            subset = [r for r in rows if r['edge_id'].startswith(component + ':')]
            labels = [r['edge_id'].split(':', 1)[1] for r in subset if r['model_id'] == models[0]]
            for mi, model in enumerate(models):
                rr = [r for r in subset if r['model_id'] == model]
                x = np.arange(len(rr), dtype=float)
                s = model_semantics(p, model)
                ax.scatter(x + (mi - 2) * 0.055, [r['mean'] for r in rr], marker=s['marker'], c=s['color'], s=p.style['marker']['size_pt'] ** 2, label=MODEL_LABELS[model], zorder=3)
                for j, r in enumerate(rr):
                    vals = [v['q'] for v in r['seed_points']]
                    ax.scatter(x[j] + (mi - 2) * 0.055 + np.linspace(-0.018, 0.018, len(vals)), vals, marker='o', facecolors='white', edgecolors='#777777', linewidths=0.45, s=10, zorder=2)
            ax.set(title=component.title(), xticks=np.arange(4), xticklabels=labels, ylim=(0, 1), ylabel='Anchor-Q')
            ax.tick_params(axis='x', labelrotation=35)
        shared_legend(fig, axs, p, rows=2)
        fig._phonetics_resolution = {'kind': 'anchor_q_transitions', 'model_order': models, 'seed_points': 5}
        fig, axs = layout(1, p, columns=1, legend_rows=2, width_mm=180)
        figs.append(fig)
        ax = axs[0]
        key, field, n = ('transition_shares', 'edge_id', 12)
        rs = anchor_plot[key]
        for ax in axs:
            for ci, comp in enumerate(components):
                values = []
            first_valid = next((m for m in models if all((r['share'] is not None for r in rs if r['model_id'] == m))), None)
            for mi, model in enumerate(models):
                names = [r[field] for r in rs if r['model_id'] == model]
                vals = [next((r['share'] for r in rs if r['model_id'] == model and r[field] == name)) for name in names]
                if len(vals) != n or any((v is None for v in vals)):
                    continue
                if n == 12:
                    bottom = 0.0
                    for ei, (name, val) in enumerate(zip(names, vals)):
                        component = name.split(':', 1)[0]
                        shade = ('#555555', '#999999', '#cccccc')[('initial', 'final', 'tone').index(component)]
                        hatch = ('', '/', '\\', 'x')[ei % 4]
                        ax.bar(mi, val, bottom=bottom, width=0.68, color=shade, hatch=hatch, edgecolor='black', linewidth=0.25, label=f"{component.title()} {name.split(':', 1)[1]}" if model == first_valid else None)
                        bottom += val
                else:
                    b = 0
                    for ci, val in enumerate(vals):
                        ax.bar(mi, val, bottom=b, width=0.65, color=('#444444', '#888888', '#c5c5c5')[ci], hatch=('', '//', 'xx')[ci], edgecolor='black', linewidth=0.3, label=components[ci] if mi == 0 else None)
                        b += val
            ax.set_ylim(0, 1)
            ax.set_ylabel('Q share')
            ax.set_xticks(np.arange(5))
            ax.set_xticklabels([MODEL_LABELS[m] for m in models], rotation=25, ha='right')
            if n == 12:
                for mi, model in enumerate(models):
                    if any((r['model_id'] == model and r.get('status') == 'undefined_zero_total' for r in rs)):
                        ax.text(mi, 0.5, 'undefined', rotation=90, ha='center', va='center')
        shared_legend(fig, axs, p, rows=2)
        fig._phonetics_resolution = {'kind': 'anchor_q_transition_shares', 'undefined_zero_total': 'shown as absent bars'}
        fig, axs = layout(1, p, columns=1, legend_rows=1, width_mm=180)
        figs.append(fig)
        ax = axs[0]
        rs = anchor_plot['component_shares']
        first_component_valid = next((m for m in models if all((r['share'] is not None for r in rs if r['model_id'] == m))), None)
        for mi, model in enumerate(models):
            vals = [next((r['share'] for r in rs if r['model_id'] == model and r['component'] == comp)) for comp in components]
            if any((v is None for v in vals)):
                ax.text(mi, 0.5, 'undefined', rotation=90, ha='center', va='center')
                continue
            bottom = 0.0
            for ci, (comp, val) in enumerate(zip(components, vals)):
                ax.bar(mi, val, bottom=bottom, width=0.65, color=('#444444', '#888888', '#c5c5c5')[ci], hatch=('', '//', 'xx')[ci], edgecolor='black', linewidth=0.3, label=comp.title() if model == first_component_valid else None)
                bottom += val
        ax.set(ylim=(0, 1), ylabel='Q share', xticks=range(5), xticklabels=[MODEL_LABELS[m] for m in models])
        shared_legend(fig, axs, p)
        fig._phonetics_resolution = {'kind': 'anchor_q_component_shares', 'undefined_zero_total': 'shown as absent bars'}
        curves = dimsim_plot['curves']
        bounds = dimsim_plot['boundaries']
        fig, axs = layout(10, p, columns=2, legend_rows=3, width_mm=180)
        figs.append(fig)
        for host in axs:
            pos = host.get_position()
            host.set_position([pos.x0, pos.y0, pos.width - 0.06, pos.height])
        count_rows = dimsim_plot.get('mode_counts', [])
        max_bin_draws = max((pt['draw_count'] for row in curves for pt in row['seed_points'] if pt['draw_count'] is not None), default=1)
        for mi, model in enumerate(models):
            for col, mode in enumerate(('dimsim-natural', 'dimsim-balanced')):
                ax = axs[mi * 2 + col]
                cx = ax.twinx()
                ax.set_zorder(cx.get_zorder() + 1)
                ax.patch.set_visible(False)
                s = model_semantics(p, model)
                cur = [r for r in curves if r['model_id'] == model and r['mode'] == mode]
                x = np.arange(13)
                bar = next((r for r in count_rows if r['mode'] == mode), {})
                draws = bar.get('draw_count', 0)
                unique = bar.get('unique_pair_count', 0)
                bin_draws = [r['seed_points'][0]['draw_count'] for r in cur]
                bin_unique = [r['seed_points'][0]['unique_pair_count'] for r in cur]
                cx.bar(np.arange(len(cur)), bin_draws, width=0.76, color='#e6e6e6', edgecolor='none', zorder=0)
                cx.bar(np.arange(len(cur)), bin_unique, width=0.76, color='#a0a0a0', edgecolor='none', zorder=1)
                for j, r in enumerate(cur):
                    if r['mean_cosine'] is not None:
                        ax.plot([j], [r['mean_cosine']], marker=s['marker'] if r['supported'] else 'o', markerfacecolor=s['color'] if r['supported'] else 'none', markeredgecolor=s['color'], linestyle='none', markersize=p.style['marker']['size_pt'])
                for si in range(5):
                    for j in range(len(cur) - 1):
                        a, b = (cur[j]['seed_points'][si], cur[j + 1]['seed_points'][si])
                        if cur[j]['supported'] and cur[j + 1]['supported'] and (a['mean_cosine'] is not None) and (b['mean_cosine'] is not None):
                            ax.plot([j, j + 1], [a['mean_cosine'], b['mean_cosine']], color='#aaaaaa', linewidth=p.style['line']['reference_pt'])
                bd = next((r for r in bounds if r['model_id'] == model and r['mode'] == mode))
                bx = (bd['median'] - 5) / 10 if bd['median'] is not None else None
                for j in range(len(cur) - 1):
                    a, b = (cur[j], cur[j + 1])
                    if a['supported'] and b['supported'] and (a['mean_cosine'] is not None) and (b['mean_cosine'] is not None):
                        if bx is not None and j < bx < j + 1:
                            by = np.interp(bx, [j, j + 1], [a['mean_cosine'], b['mean_cosine']])
                            ax.plot([j, bx], [a['mean_cosine'], by], color='#171717', linewidth=p.style['line']['data_pt'], linestyle='-')
                            ax.plot([bx, j + 1], [by, b['mean_cosine']], color='#171717', linewidth=p.style['line']['data_pt'], linestyle='--')
                        else:
                            solid = bx is None or bx >= j + 1
                            ax.plot([j, j + 1], [a['mean_cosine'], b['mean_cosine']], color='#171717', linewidth=p.style['line']['data_pt'], linestyle='-' if solid else '--')
                if bx is not None:
                    ax.axvline(bx, color='#171717', linewidth=p.style['line']['reference_pt'])
                    j = int(np.floor(bx))
                    if 0 <= j < len(cur) - 1 and cur[j]['supported'] and cur[j + 1]['supported'] and (cur[j]['mean_cosine'] is not None) and (cur[j + 1]['mean_cosine'] is not None):
                        by = np.interp(bx, [j, j + 1], [cur[j]['mean_cosine'], cur[j + 1]['mean_cosine']])
                        ax.plot([bx], [by], marker='D', color='#171717', markersize=p.style['marker']['size_pt'])
                cx.set_ylim(0, max_bin_draws * 1.05)
                cx.set_yticks([0, max_bin_draws])
                cx.set_yticklabels(['0', str(max_bin_draws)])
                ax.set_ylim(-1, 1)
                ax.set_xlim(-0.5, 12.5)
                ax.set_xticks(range(13))
                ax.set_xticklabels([str(x) for x in range(5, 120, 10)] + ['[120,∞)'], rotation=45)
                ax.set_title(f'{MODEL_LABELS[model]} · {mode}')
                ax.set_ylabel('Cosine similarity' if col == 0 else '')
                cx.set_ylabel('Pair draws' if col == 1 else '')
                if mi == 4:
                    ax.set_xlabel('DIMSIM distance (bin centers)', labelpad=-2)
        from matplotlib.lines import Line2D
        from matplotlib.patches import Patch
        fig.legend(handles=[Line2D([], [], color='#171717', label='Mean cosine'), Line2D([], [], color='#aaaaaa', label='Seed curves'), Patch(facecolor='#e6e6e6', label='All draws'), Patch(facecolor='#a0a0a0', label='Unique pairs'), Line2D([], [], color='#171717', linestyle='--', marker='D', label='Exploratory boundary')], labels=['Mean cosine', 'Seed curves', 'All draws', 'Unique pairs', 'Exploratory boundary'], loc='upper center', bbox_to_anchor=(0.5, 0.995), ncol=5, frameon=False)
        fig._phonetics_resolution = {'kind': 'dimsim_geometry', 'support_gaps_not_bridged': True, 'boundary': 'exploratory median of valid seed thresholds'}
        fig, axs = layout(2, p, columns=2, legend_rows=2, width_mm=180)
        figs.append(fig)
        rho = dimsim_plot.get('rho', [])
        for ax, mode in zip(axs, ('dimsim-natural', 'dimsim-balanced')):
            rr = [r for r in rho if r['mode'] == mode]
            for mi, model in enumerate(models):
                r = next((v for v in rr if v['model_id'] == model), None)
                if not r or r.get('mean') is None:
                    continue
                s = model_semantics(p, model)
                ax.scatter([mi], [r['mean']], marker=s['marker'], c=s['color'], label=MODEL_LABELS[model])
                vals = [x['rho'] for x in r['seed_values'] if x['rho'] is not None]
                ax.scatter([mi] * len(vals), vals, marker='.', c='#aaaaaa', s=10)
            ax.axhline(0, color='#777777', lw=p.style['line']['reference_pt'])
            ax.set(title=mode, xlim=(-0.5, 4.5), xticks=range(5), xticklabels=[MODEL_LABELS[m] for m in models], ylabel='Spearman ρ')
            ax.tick_params(axis='x', labelrotation=25)
        shared_legend(fig, axs, p, rows=2)
        fig._phonetics_resolution = {'kind': 'dimsim_correlation', 'five_seed_points': True}
        return figs
    except Exception:
        for fig in figs:
            plt.close(fig)
        raise

def render_evaluation(anchor_output, dimsim_output, output_dir, source_digest, *, protocol=None):
    """Render five PNG/EPS pairs after validating both complete output manifests."""
    aq = load_evaluation_outputs(anchor_output)
    ds = load_evaluation_outputs(dimsim_output)
    check_paired_outputs(aq, ds)
    if not isinstance(source_digest, str) or not re.fullmatch('[0-9a-f]{64}', source_digest):
        raise ValueError('source_digest must be an explicit lowercase SHA-256 digest')
    p = protocol or load_protocol()
    apply_style(p)
    out = Path(output_dir)
    guard_report_output(out)
    stems = FIGURE_STEMS
    targets = [out / f'{stem}.{ext}' for stem in stems for ext in ('png', 'eps', 'metadata.json')]
    if any((x.exists() for x in targets)):
        raise FileExistsError('Refusing to overwrite evaluation figure exports')
    dplot = dict(ds['plot_data'])
    dplot['rho'] = ds['tables'].get('model_rho', [])
    figs = build_figures(aq['plot_data'], dplot, p)
    saved = []
    try:
        for fig, stem, fid in zip(figs, stems, ('anchor_q_transitions', 'transition_shares', 'component_shares', 'dimsim_geometry', 'dimsim_correlation')):
            saved.extend(export_figure(fig, out, stem, fid, p, scientific_provenance={'source_digest': source_digest, 'anchor_q_output': str(aq['root']), 'dimsim_output': str(ds['root']), 'anchor_q_manifest_sha256': manifest_digest(aq['root'] / 'manifest.json'), 'dimsim_manifest_sha256': manifest_digest(ds['root'] / 'manifest.json')}))
            saved.append(out / f'{stem}.metadata.json')
        return saved
    finally:
        for fig in figs:
            plt.close(fig)


### Presentation run


In [ ]:
def write_evaluation_report(project_root, run_id, *, anchor_q_output, dimsim_output, random_control_output, source_hashes, source_digest, resolved_config=None, protocol=None):
    """Save tables and PNG/EPS figures without checkpoints, extraction or scoring."""
    run_id = safe_run_id(run_id)
    if not isinstance(source_digest, str) or re.fullmatch('[0-9a-f]{64}', source_digest) is None:
        raise ValueError('source_digest must be an explicit lowercase SHA-256 digest')
    inputs = [Path(path).resolve(strict=True) for path in (anchor_q_output, dimsim_output, random_control_output)]
    target = Path(project_root).resolve() / 'outputs' / 'contextual-evaluation-report' / run_id
    ensure_output_outside_inputs(target, inputs)
    if target.exists():
        raise FileExistsError(f'refusing to overwrite existing output directory: {target}')
    guard_report_output(target)
    if not isinstance(source_hashes, dict) or not source_hashes or any((not isinstance(value, str) or re.fullmatch('[0-9a-f]{64}', value) is None for value in source_hashes.values())):
        raise ValueError('explicit source code SHA-256 map is required')
    if resolved_config is not None and (not isinstance(resolved_config, dict)):
        raise ValueError('resolved_config must be a mapping')
    aq, ds = (load_evaluation_outputs(inputs[0]), load_evaluation_outputs(inputs[1]))
    check_paired_outputs(aq, ds)
    random = load_random_control_summary(inputs[2])
    bundle_hash = aq['manifest']['scientific_input_pins']['bundle_manifest_sha256']
    if random['manifest']['scientific_input_pins']['bundle_manifest_sha256'] != bundle_hash:
        raise ValueError('random control and learned models use different evaluation bundles')
    p = protocol or load_protocol()
    apply_style(p)
    config = {**(resolved_config or {}), 'anchor_q_output': str(inputs[0]), 'dimsim_output': str(inputs[1]), 'random_control_output': str(inputs[2]), 'source_digest': source_digest, 'style_sha256': p.style_hash, 'registry_sha256': p.registry_hash, 'formats': list(p.style['export']['formats']), 'png_dpi': p.style['export']['dpi']}
    pins = {'evaluation_bundle_manifest_sha256': bundle_hash, 'anchor_q_manifest_sha256': file_sha256(inputs[0] / 'manifest.json'), 'dimsim_manifest_sha256': file_sha256(inputs[1] / 'manifest.json'), 'random_control_manifest_sha256': file_sha256(inputs[2] / 'manifest.json'), 'model_ids': aq['manifest']['scientific_input_pins']['model_ids'], 'seeds': aq['manifest']['scientific_input_pins']['seeds']}
    with tempfile.TemporaryDirectory(prefix='evaluation-tables-') as directory:
        stage = Path(directory)
        table_paths = render_evaluation_tables(aq['tables'], ds['tables'], stage / 'tables')
        expected = {'config.json', 'data/random_control_summary.json', 'diagnostics/summary.json'}
        expected.update((f'tables/{path.name}' for path in table_paths))
        expected.update((f'figures/{stem}.{suffix}' for stem in FIGURE_STEMS for suffix in ('png', 'eps', 'metadata.json')))
        with presentation_run(project_root, 'contextual-evaluation-report', run_id, input_roots=inputs, config=config, source_hashes=source_hashes, expected_files=expected, scientific_pins=pins, profile='contextual_evaluation_report', figure_exports_on_complete='complete') as root:
            for path in table_paths:
                shutil.copyfile(path, root / 'tables' / path.name)
            write_data_json(root / 'data/random_control_summary.json', random['report'])
            render_evaluation(inputs[0], inputs[1], root / 'figures', source_digest, protocol=p)
            write_data_json(root / 'diagnostics/summary.json', {'scope': 'presentation', 'figure_count': len(FIGURE_STEMS), 'table_count': len(table_paths), 'evaluation_bundle_manifest_sha256': bundle_hash})
    return target


## 4. Select and verify artifacts

Each input manifest and artifact checksum is checked. Q and DIMSIM must describe the same bundle, models, seeds and checkpoints; the random control must use the same evaluation bundle. No source training directories are opened.


In [ ]:
def resolve_evaluation_runs(output_root, run_ids=None):
    """Resolve explicit runs or require exactly one completed candidate per analysis."""
    output_root = Path(output_root).resolve(strict=True)
    analyses = ("contextual-anchor-q", "contextual-dimsim", "contextual-random-control")
    if run_ids is not None and (not isinstance(run_ids, dict) or set(run_ids) != set(analyses)):
        raise ValueError("EVALUATION_RUN_IDS must specify all three analysis IDs")
    paths = {}
    for analysis in analyses:
        parent = output_root / analysis
        if run_ids is not None:
            paths[analysis] = parent / safe_run_id(run_ids[analysis])
            continue
        candidates = []
        for candidate in sorted(parent.iterdir()):
            manifest_path = candidate / "manifest.json"
            if not candidate.is_dir() or not manifest_path.is_file():
                continue
            manifest = read_finite_json(manifest_path)
            if (manifest.get("status") == "complete" and manifest.get("analysis") == analysis
                    and manifest.get("scope") == "numerical_outputs" and manifest.get("run_id") == candidate.name):
                candidates.append(candidate)
        if len(candidates) != 1:
            raise ValueError(
                f"{analysis}: found {len(candidates)} completed runs; set EVALUATION_RUN_IDS explicitly. "
                f"Candidates: {[path.name for path in candidates]}"
            )
        paths[analysis] = candidates[0]
    return paths


In [ ]:
INPUT_PATHS = resolve_evaluation_runs(EVALUATION_OUTPUT_ROOT, EVALUATION_RUN_IDS)
PROTOCOL = load_protocol()
apply_style(PROTOCOL)
SOURCE_HASHES = {
    **BUILD_METADATA["source_sha256"],
    **BUILD_METADATA["resource_sha256"],
    "visualization_notebook_builder": BUILD_METADATA["generator_sha256"],
}
SOURCE_DIGEST = hashlib.sha256(json.dumps(BUILD_METADATA, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()
FONT_PROVENANCE = {}
for weight in ("normal", "bold"):
    filename = Path(font_manager.findfont(font_manager.FontProperties(family="Arial", weight=weight), fallback_to_default=False))
    FONT_PROVENANCE[weight] = {"path": str(filename), "sha256": file_sha256(filename)}
for analysis, path in INPUT_PATHS.items():
    print(analysis, path)


## 5. Build and save the report

Save Anchor-Q transitions, Transition shares, Component shares, DIMSIM geometry and DIMSIM correlation, with companion metadata and tables. Inputs remain unchanged. Existing or failed presentation directories are not overwritten; select a new ID for another export.


In [ ]:
REPORT_PATH = write_evaluation_report(
    OUTPUT_PROJECT_ROOT,
    PRESENTATION_ID,
    anchor_q_output=INPUT_PATHS["contextual-anchor-q"],
    dimsim_output=INPUT_PATHS["contextual-dimsim"],
    random_control_output=INPUT_PATHS["contextual-random-control"],
    source_hashes=SOURCE_HASHES,
    source_digest=SOURCE_DIGEST,
    resolved_config={
        "build_metadata": BUILD_METADATA,
        "selected_run_ids": {name: path.name for name, path in INPUT_PATHS.items()},
        "fonts": FONT_PROVENANCE,
    },
    protocol=PROTOCOL,
)
print("Saved report:", REPORT_PATH)
print("Figures:", REPORT_PATH / "figures")
print("Tables:", REPORT_PATH / "tables")
